# 🧠 Notebook 3 — Custom CNN Training
## Aerial Bird vs Drone Classification

**Objectives:**
- Build a custom CNN from scratch
- Train with EarlyStopping & ReduceLROnPlateau
- Plot training curves
- Evaluate on test set

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

from src.utils import set_seed, ensure_dirs
from src.config import PLOTS_DIR, SAVED_MODELS_DIR
from src.data_preprocessing import get_all_generators
from src.custom_cnn import build_custom_cnn, compile_model, train_custom_cnn
from src.evaluate import evaluate_model

set_seed()
ensure_dirs(PLOTS_DIR, SAVED_MODELS_DIR)
print('✅ Ready to train Custom CNN')

## 1. Model Architecture

In [ ]:
model = build_custom_cnn()
model.summary()

total_params = model.count_params()
print(f'\nTotal Parameters: {total_params:,}')

## 2. Compile & Load Data

In [ ]:
model = compile_model(model)
train_gen, valid_gen, test_gen = get_all_generators()

## 3. Train

In [ ]:
# Full training pipeline
model, history = train_custom_cnn(train_gen, valid_gen, epochs=50)

## 4. Training Curves

In [ ]:
from src.utils import plot_training_history
import os
plot_training_history(
    history,
    model_name='Custom CNN',
    save_path=os.path.join(PLOTS_DIR, 'custom_cnn_history.png')
)

## 5. Evaluate on Test Set

In [ ]:
results = evaluate_model(model, test_gen, model_name='Custom CNN')
print('\nFinal Test Metrics:')
for k, v in results.items():
    if k != 'model':
        print(f'  {k.upper():<12}: {v:.4f}')

## 6. Grad-CAM Visualization

In [ ]:
from src.evaluate import display_gradcam
from src.config import TRAIN_DIR

# Grad-CAM on a sample image
sample_img = os.listdir(os.path.join(TRAIN_DIR, 'bird'))[0]
sample_path = os.path.join(TRAIN_DIR, 'bird', sample_img)

display_gradcam(
    img_path=sample_path,
    model=model,
    last_conv_layer='conv4_2',   # Last conv layer in custom CNN
    save_path=os.path.join(PLOTS_DIR, 'gradcam_custom_cnn.png')
)